# `ingestion/crossref.py` — Validation Walkthrough

Validates prerequisite and cross-reference extraction against the FE 501s manual.

Two types are extracted from chunk text:
- **Prerequisites** (`Condition:` lines) — state that must be true before a procedure.
  ~34 occurrences across the manual.  Stored as raw strings; the retrieval layer uses
  them as semantic search queries to surface prerequisite sections.
- **References** (`(p. NN)` page citations) — resolved to section numbers via a range
  lookup against the TOC.  Appear consistently in **Preparatory work** sections as the
  machine-readable pointer to a prerequisite procedure, e.g.:
  ```
  ‒ Remove the fuel tank.
   (p. 108)
  ```
  83 preparatory chunks carry references; 161 individual section-number links total.
  References are restricted to preparatory chunks — the same `(p. NN)` pattern in main
  work chunks points to the special tools appendix, not a prerequisite procedure.

In [1]:
import sys
sys.path.insert(0, "..")

import pypdf
from ingestion.parse import parse_manual, parse_toc
from ingestion.index import build_chunks
from ingestion.crossref import annotate_chunks

PDF = "../manuals/FE_501s_2026_US_en_Bundle_RM_069969-000001_05sq_m1du.pdf"

pages  = parse_manual(PDF, extract_images=False)
toc    = parse_toc(pypdf.PdfReader(PDF))
chunks = build_chunks(pages, toc, "FE_501s_2026")
annotate_chunks(chunks, toc)

with_prereqs = [c for c in chunks if c.prerequisites]
with_refs    = [c for c in chunks if c.references]

print(f"Total chunks:              {len(chunks)}")
print(f"Chunks with prerequisites: {len(with_prereqs)}")
print(f"Chunks with references:    {len(with_refs)}")

Total chunks:              600
Chunks with prerequisites: 34
Chunks with references:    82


## 1. Prerequisites

`Condition:` lines describe the state the bike must be in before a procedure begins.
They are the primary dependency signal the agent uses to order repair steps.

In [2]:
print(f"{'Section':<10} {'Prerequisite'}")
print("-" * 80)
for c in with_prereqs:
    for prereq in c.prerequisites:
        print(f"{c.section:<10} {prereq[:70]}")

Section    Prerequisite
--------------------------------------------------------------------------------
5.3        Ambient temperature: < 20 °C (< 68.0 °F)
6.10       The fork legs have been removed
6.15       The fork legs have been disassembled
6.9        The fork legs have been removed
8.4        Frame protector removed on left and right
9.12       The shock absorber is removed
9.13       The shock absorber is removed
9.16       The damper has been disassembled
9.17       The shock absorber is removed
12.12      The fuel tank is completely full, Ensure that the battery voltage does
15.7       The 12 V battery must be fully functional and completely charged.
17.10      Motorcycle is stationary
17.7       Motorcycle is stationary
17.8       Motorcycle is stationary
17.9       Motorcycle is stationary
18.4.25    The valves are disassembled and the exhaust flange is removed
18.4.38    Gearbox has been dismantled
18.5.7     The timing chain tensioner and camshafts have been removed (see

## 2. Prerequisite chains

Some sections share the same prerequisite, forming a dependency chain.
For example, several shock absorber sub-procedures all require the shock absorber
to be removed first.  The retrieval layer can use these to pull in upstream sections
automatically when a user asks about a downstream procedure.

In [3]:
from collections import defaultdict

prereq_to_sections: dict[str, list[str]] = defaultdict(list)
for c in with_prereqs:
    for p in c.prerequisites:
        # Normalise minor variations (capitalisation, trailing punctuation)
        prereq_to_sections[p.lower().rstrip(".,")].append(c.section)

# Show prerequisites shared by more than one section
shared = {k: v for k, v in prereq_to_sections.items() if len(v) > 1}
print(f"Prerequisites shared by multiple sections: {len(shared)}")
print()
for prereq, sections in sorted(shared.items(), key=lambda x: -len(x[1])):
    print(f"{sections}")
    print(f"  → {prereq[:80]}")

Prerequisites shared by multiple sections: 5

['17.10', '17.7', '17.8', '17.9']
  → motorcycle is stationary
['21.2', '21.3', '21.4', '21.6']
  → the engine is cold
['9.12', '9.13', '9.17']
  → the shock absorber is removed
['20.2', '26.2', '26.3']
  → diagnostic tool is connected and active
['6.10', '6.9']
  → the fork legs have been removed


## 3. Page-number cross-references

Preparatory work sections list prerequisite steps as bullet points, each followed by a
`(p. NN)` citation on the next line.  The page number is resolved to a section using a
range lookup against the TOC — the section whose start page is the largest value ≤ N.

In [4]:
# Show the raw text and resolved references for a few preparatory chunks
samples = [c for c in chunks if c.phase == "preparatory" and c.references][:4]

for c in samples:
    print(f"=== {c.section} [{c.phase}] ===")
    print(c.text.strip())
    print(f"\nResolved references:")
    for sec in c.references:
        title = next((e["title"] for e in toc if e["number"] == sec), "?")
        print(f"  {sec}  {title}")
    print()

=== 6.11 [preparatory] ===
Preparatory work
‒ Disassemble the fork legs.
 (p. 22)

Resolved references:
  6.10  Disassembling the fork legs

=== 6.12 [preparatory] ===
Preparatory work
‒ Disassemble the fork legs.
 (p. 22)
‒ Disassemble the cartridge.
 (p. 25)

Resolved references:
  6.10  Disassembling the fork legs
  6.11  Disassembling the cartridge

=== 6.13 [preparatory] ===
Preparatory work
‒ Disassemble the fork legs.
 (p. 22)
‒ Disassemble the cartridge.
 (p. 25)

Resolved references:
  6.10  Disassembling the fork legs
  6.11  Disassembling the cartridge

=== 6.14 [preparatory] ===
Preparatory work
‒ Disassemble the fork legs.
 (p. 22)
‒ Disassemble the cartridge.
 (p. 25)
‒ Disassemble the piston rod.
 (p. 27)
Douglas Raffle, raffled@gmail.com, 069969/074383
Fork, triple clamp 6
31

Resolved references:
  6.10  Disassembling the fork legs
  6.11  Disassembling the cartridge
  6.12  Disassembling the piston rod



## 4. Why references are restricted to preparatory chunks

The `(p. NN)` pattern appears in **main work** chunks too, but there it refers to the
special tools appendix (pages 354+), not a prerequisite procedure.  The two uses are
structurally identical — only the context (phase) distinguishes them.

In [5]:
import re

page_ref_re = re.compile(r'\(p\.?\s*(\d+)\)', re.IGNORECASE)

# Show the dual meaning: same pattern, different purpose
print("Preparatory chunk — (p. NN) = prerequisite procedure:")
prep = next(c for c in chunks if c.section == "7.4" and c.phase == "preparatory")
for line in prep.text.splitlines():
    if line.strip():
        print(f"  {line}")
print(f"  → resolved references: {prep.references}")

print()
print("Main work chunk — (p. NN) = special tool catalog page:")
main = next(c for c in chunks if c.section == "6.10" and c.phase == "")
lines_with_refs = [l for l in main.text.splitlines() if page_ref_re.search(l)]
for line in lines_with_refs[:4]:
    print(f"  {line.strip()}")
print(f"  → references suppressed (phase != preparatory)")

Preparatory chunk — (p. NN) = prerequisite procedure:
  Preparatory work
  ‒ Remove the seat.
   (p. 106)
  ‒ Remove the fuel tank.
   (p. 108)
  Douglas Raffle, raffled@gmail.com, 069969/074383
  7 Handlebar, controls
  52
  → resolved references: ['12.4', '12.7']

Main work chunk — (p. NN) = special tool catalog page:
  (p. 363)
  (p. 362)
  → references suppressed (phase != preparatory)


## 5. Metadata schema

Verify that `prerequisites` and `references` appear as expected on annotated chunks.
Section 6.12 preparatory is a good spot-check — it references both 6.10 and 6.11.

In [6]:
sample = next(c for c in chunks if c.section == "6.12" and c.phase == "preparatory")

print(f"Section      : {sample.section} — {sample.section_title}")
print(f"Phase        : {sample.phase}")
print(f"Prerequisites: {sample.prerequisites}")
print(f"References   : {sample.references}")
for sec in sample.references:
    title = next((e["title"] for e in toc if e["number"] == sec), "?")
    print(f"  {sec}  {title}")
print(f"\nText:\n{sample.text.strip()}")

Section      : 6.12 — Disassembling the piston rod
Phase        : preparatory
Prerequisites: []
References   : ['6.10', '6.11']
  6.10  Disassembling the fork legs
  6.11  Disassembling the cartridge

Text:
Preparatory work
‒ Disassemble the fork legs.
 (p. 22)
‒ Disassemble the cartridge.
 (p. 25)
